# **EXPERIMENT NOTEBOOK**

---

In this notebook you can run experiments on the LIBERO environment using the ActionJEPA VLA trained by us or your version of ActionJEPA. Follow ALL the instructions in the notebook!


## **IMPORT SECTION**

In [1]:
import sys
import os
current_dir = os.getcwd()
libero_path = os.path.join(current_dir, "LIBERO")
if libero_path not in sys.path:
    sys.path.insert(0, libero_path)
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)


from libero.libero import benchmark
from libero.libero.envs.env_wrapper import ControlEnv
from libero.libero.utils import get_libero_path
import numpy as np
import imageio
import torch
from collections import deque
import random
from model.MLPActionJEPA import MLPActionJEPA
from model.TransformerActionJEPA import TransformerActionJEPA
from tqdm.notebook import tqdm
import cv2
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using the device: {device}")

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/cyberm/miniconda3/envs/myenv/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/home/cyberm/miniconda3/envs/myenv/lib/python3.10/site-packages/wandb/util.py:118: SentryHubDeprecationWarning: `sentry_sdk.Hub` is deprecated and will be removed in a future major release. Please consult our 1.x to 2.x migration guide for details on how to migrate `Hub` usage to the new API: https://docs.sentry.io/platforms/python/

Using the device: cuda


## **HYPERPARAMETER CONFIGURATION**

In [2]:
results_dir_path = './results/2026_04_23__16_44'

In [3]:
RENDER_CAMERA = "agentview" 

checkpoints_path = './checkpoints'

model_path = os.path.join(results_dir_path, 'best_model.pth')
experiment_config_path = os.path.join(results_dir_path, 'experiment_config.json')
metrics_path = os.path.join(results_dir_path, 'metrics.csv')

with open(experiment_config_path, 'r') as f:
    experiment_config = json.load(f)

TASK_NAME = experiment_config["hyperparameters"]['dataset_type']
if TASK_NAME == 'all':
    TASK_NAME = random.choice(['libero_10', 'libero_90', 'libero_goal', 'libero_spatial', 'libero_object'])

POLICY_TYPE = experiment_config["hyperparameters"]['policy']
INTERPOLATION_TYPE = experiment_config["hyperparameters"]['interpolation_type']

print(f" Camera initialized on: {RENDER_CAMERA}")
print(f" Task suite: {TASK_NAME}")
print(f" Policy type: {POLICY_TYPE}")
print(f" Interpolation: {INTERPOLATION_TYPE}")


 Camera initialized on: agentview
 Task suite: libero_goal
 Policy type: transformer
 Interpolation: linear


## **MODEL DEFINITION SECTION**

In [4]:
# Path of the models V-JEPA 2 Encoder, CLIP Encoder and V-JEPA 2 AC Predictor
vjepa_path = os.path.join(checkpoints_path,"facebook/vjepa2-vitg-fpc64-256")
vjepa_pred_path = os.path.join(checkpoints_path,"facebook/jepa-wms/vjepa2_ac_droid.pth.tar/vjepa2_ac_droid.pth.tar")
clip_path = os.path.join(checkpoints_path,"openai/clip-vit-large-patch14")

# Path of your model to test
print("Loading model...")
model = torch.load(model_path, map_location=device)
checkpoint = model['model_state_dict']
model_config = model['config']

if POLICY_TYPE == 'mlp':
    model = MLPActionJEPA(
        vjepa_encoder_path=vjepa_path,
        vjepa_predictor_path=vjepa_pred_path,
        clip_model_path=clip_path,
        num_frames=model_config['num_frames'],
        use_backbone = True,
        device=device
    ).to(device)
else:
    model = TransformerActionJEPA(
        vjepa_encoder_path=vjepa_path,
        vjepa_predictor_path=vjepa_pred_path,
        clip_model_path=clip_path,
        num_frames=model_config['num_frames'],
        use_backbone = True,
        device=device
    )

model.load_state_dict(checkpoint, strict=False)
model.to(device)
model.eval()

Loading model...


Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: ./checkpoints/openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TransformerActionJEPA(
  (vision_backbone): VJEPAEncoder(
    (model): VJEPA2Model(
      (encoder): VJEPA2Encoder(
        (embeddings): VJEPA2Embeddings(
          (patch_embeddings): VJEPA2PatchEmbeddings3D(
            (proj): Conv3d(3, 1408, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
        )
        (layer): ModuleList(
          (0-39): 40 x VJEPA2Layer(
            (norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
            (attention): VJEPA2RopeAttention(
              (query): Linear(in_features=1408, out_features=1408, bias=True)
              (key): Linear(in_features=1408, out_features=1408, bias=True)
              (value): Linear(in_features=1408, out_features=1408, bias=True)
              (proj): Linear(in_features=1408, out_features=1408, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (drop_path): Identity()
            (norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
         

## **INFERENCE LOOP**

In [25]:
import h5py
file_path = './LIBERO/libero/datasets/libero_goal/turn_on_the_stove_demo.hdf5'
with h5py.File(file_path, 'r') as f:
    demo = f['data']['demo_1']
    actions = demo['actions'][:]
    frames_demo = demo['obs']['agentview_rgb'][:]
    joint_states = demo['obs']['joint_states'][:]


print(frames_demo.shape)


(96, 128, 128, 3)


In [27]:
print("="*50 + "\n")
print(f"[info] RENDER_CAMERA: {RENDER_CAMERA}\n")

benchmark_dict = benchmark.get_benchmark_dict() # dictionary of the type {"<task_suite_name>": <task_suite_class>} (Ex. "libero_spatial": <class 'libero.libero.benchmark.LIBERO_SPATIAL'>)
task_suite_name = TASK_NAME # can also choose libero_spatial, libero_object, etc.
task_suite = benchmark_dict[task_suite_name]() # task suite is a class that represents a set of different tasks, we'll retrieve a specific task with the task_id
n_tasks = task_suite.get_num_tasks() # number of tasks in the selected benchmark
print(f"[info] The benchmark {TASK_NAME} has {n_tasks} tasks.")


# retrieve a random task
#for i in range(n_tasks):
task_id = 7 # id of the task to retrieve
task = task_suite.get_task(task_id) # task is the object with all the information about the specific task
task_name = task.name # the name of the task as "KITCHEN_SCENE_1_put_the_black_bowl_at_the_front_on_the_coffee_table"
task_description = task.language # instruction in natural language ("Es. put the black bowl at the front on the coffee table")
print(f"\n[info] retrieving task {task_id} from suite {task_suite_name}\n")
print(f"[info] task description: {task_description}\n")

# retrieve the BDDL (Behavioral Design Definition Language) file for the task, it is a file containing all the informations about objects and initial and goal state
# (Ex. On Bowl1 Table1 means that the bowl 1 is on the table at initial state for example)
# get_libero_path("bddl_files"): find the directory where BDDL files are stored in your pc (Ex. /home/user/Desktop/LIBERO/libero/libero/./bddl_files )
# task.problem_folder: the folder containing the BDDL file for the specific task (Ex. libero_spatial)
# task.bddl_file: the name of the BDDL file for the specific task (Ex. pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate.bddl)
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
print(f"[info] BDDL file for the task: {task_bddl_file}\n")


# Args for environment initialization
env_args = {
    "bddl_file_name": task_bddl_file, # path of the BDDL file
    "camera_heights": 128,
    "camera_widths": 128,
    "camera_names": ["agentview"],
    "has_renderer": False, # If True, open the MuJoCo screen to render the env
    "has_offscreen_renderer": True, # If True, save images rendered to create a video
    "use_camera_obs": True, # If True, the "obs" will include camera observations (e.g., RGB images) that can be used to create a video.
    "render_camera": RENDER_CAMERA, # the camera name used for rendering and saving video
    "controller": "OSC_POSE",
    "control_freq": 20
}

env = ControlEnv(**env_args) # create the env class
env.seed(0) # set a seed for reproducibility
env.reset() # reset the scene and bring to initial state

# init_states is a list of 50 possible different intial states
# each init_state is a vector of state formed by joint positions, objects positions and orientations, velocities ...
init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = random.randint(0, 49) # Among all 50 initial states you can spawn objects in different ways choosing init_state_id (it can be a int number in [0,49])
env.set_init_state(init_states[init_state_id]) # set the init_state chosen

# Invece di env.action_space, usa:
print(f"Controller type: {env.env.robots[0].controller.name}")
print("="*50 + "\n")

window_size = model.num_frames # the number of frames needed to make inference with the loaded model ActionJEPA

interpolation_dict= {
    "nearest": cv2.INTER_NEAREST,
    "linear": cv2.INTER_LINEAR,
    "area": cv2.INTER_AREA,
    "cubic": cv2.INTER_CUBIC
}

# the get method return the value corresponding to the key INTERPOLATION TYPE, otherwise it return INTER CUBIC method
interpolation = interpolation_dict.get(INTERPOLATION_TYPE, cv2.INTER_CUBIC)

frame_buffer = deque(maxlen=window_size) # the frame buffer for making inference with ActionJEPA
joint_buffer = deque(maxlen=model.T)

# Define the text instruction for the VLA
text_input = task_description

print(f"Task: {task_description}")

init_action = [0.] * 7 # a start action needed for the first step, to have the first observation from the environment
obs, _, _, _ = env.step(init_action) # a first step with a zero action to observe the first frame

for i in range(window_size):
    frame = frames_demo[i, :, :, :]
    frame = cv2.resize(frame, (256, 256), interpolation=interpolation)
    frame_buffer.append(frame)

for i in range(model.T):
    current_joints = joint_states[i,:]
    joint_buffer.append(current_joints)

video_frames = [] # list used to store frames for video saving

# Loop over the environment to apply actions and collect observations
# Obs is an ordered dict that have information about the ROBOT:
# - "agentview_image": the RGB image from the agent's camera => camera_heights x camera_widths x 3 (ex 1024 x 1024 x 3)
# - "robot0_joint_pos": array of vlues of the robot's joint positions
# - "robot0_joint_pos_cos (or sin)": array of cos (or sin) values of the robot's joint positions
# - "robot0_joint_vel": array of values of the robot's joint velocities
# - "robot0_eef_pos": array of values of the robot's end effector position (x,y,z)
# - "robot0_eef_quat": array of values of the robot's end effector orientation in quaternion (x,y,z,w)
# - "robot0_gripper_qpos": the gripper's position (two values for the two fingers distances from the center, 0 means fully closed)
# - "robot0_gripper_qvel": the gripper's velocity (two values for the two fingers velocities)
# And information about the object. Each object called "objectName" has two arrays:
# - "objectName_pos": array of values of the object's position (x,y,z)
# - "objectName_quat": array of values of the object's orientation in quaternion (x,y,z,w)
# - "objectName_to_robot0_eef_pos": array of values of the object's position relative to the robot's end effector (x,y,z)
# - "objectName_to_robot0_eef_quat": array of values of the object's orientation relative to the robot's end effector (x,y,z,w)
model.use_backbone = True
with torch.no_grad():
    for step in tqdm(range(window_size, len(frames_demo)), desc="🤖 Inference execution"):

        current_frame = frames_demo[step, :, :, :]
        current_frame = np.flip(current_frame, axis=0)
        current_frame = cv2.resize(current_frame, (256, 256), interpolation=interpolation)
        current_joints = joint_states[step,:]
        print(current_joints)
        
        saving_frame = cv2.resize(np.flip(obs["agentview_image"], axis=0), (256, 256), interpolation=interpolation)
        
        video_frames.append(saving_frame) # needed to save the video
        
        frame_buffer.append(current_frame) # add the current frame to the frame buffer
        joint_buffer.append(current_joints)

        vision_input = list(frame_buffer)
        joint_input = torch.from_numpy(np.array(joint_buffer)).float().unsqueeze(0).to(device)
        actor_action_seq, refiner_action_seq = model(text_input, vision_input, joint_input)

        vla_action = actor_action_seq[0, 0, :].cpu().numpy()
    
        obs, reward, done, _ = env.step(vla_action)

        if done:
            print(f"Task completed at step: {step}")
            break

    # At the end the env is closed
    env.close()

    # Save video if required to a task.mp4 file
    videoname = f"{task_suite_name}_{task_id}_{task_description.replace(' ', '_')}.mp4"
    if len(video_frames) > 0:
        imageio.mimsave(os.path.join(results_dir_path, videoname), video_frames, fps=60)
        print(f"Video saved: {os.path.join(results_dir_path, videoname)}")


[info] RENDER_CAMERA: agentview

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] The benchmark libero_goal has 10 tasks.

[info] retrieving task 7 from suite libero_goal

[info] task description: turn on the stove

[info] BDDL file for the task: /home/cyberm/Desktop/LIBERO/libero/libero/./bddl_files/libero_goal/turn_on_the_stove.bddl

[Warning]: datasets path /home/cyberm/Desktop/LIBERO/libero/libero/../datasets does not exist!
Controller type: OSC_POSE

Task: turn on the stove


🤖 Inference execution:   0%|          | 0/92 [00:00<?, ?it/s]

[-0.01643533 -0.11564526 -0.00739812 -2.40350762  0.04153304  2.16470459
  0.82045676]
[-1.23107689e-02 -1.12907140e-01  1.83561024e-03 -2.40457379e+00
  4.84856621e-02  2.15989805e+00  8.27733636e-01]
[-0.00746226 -0.11259239  0.01402184 -2.40849341  0.05448719  2.15580708
  0.83934548]
[-1.93103934e-03 -1.13450401e-01  2.88551595e-02 -2.41462468e+00
  6.10797368e-02  2.15367756e+00  8.54008919e-01]
[ 0.00408967 -0.11493661  0.04565202 -2.42222931  0.06832882  2.15343495
  0.87090867]
[ 0.01040225 -0.11661801  0.06372375 -2.43057677  0.07611467  2.15482292
  0.88922216]
[ 0.01676135 -0.11804519  0.08224315 -2.43875608  0.08404963  2.15839407
  0.90815868]
[ 0.02303469 -0.11955576  0.10072887 -2.44690146  0.09197344  2.16304598
  0.92705357]
[ 0.02909739 -0.12095517  0.1187664  -2.45499288  0.10002148  2.16882538
  0.94643007]
[ 0.03496808 -0.12221066  0.13639169 -2.46312047  0.10832957  2.17524637
  0.96629376]
[ 0.04075187 -0.123589    0.15390378 -2.47146374  0.11693937  2.18174014
 

In [28]:
print("="*50 + "\n")
print(f"[info] RENDER_CAMERA: {RENDER_CAMERA}\n")

benchmark_dict = benchmark.get_benchmark_dict() # dictionary of the type {"<task_suite_name>": <task_suite_class>} (Ex. "libero_spatial": <class 'libero.libero.benchmark.LIBERO_SPATIAL'>)
task_suite_name = TASK_NAME # can also choose libero_spatial, libero_object, etc.
task_suite = benchmark_dict[task_suite_name]() # task suite is a class that represents a set of different tasks, we'll retrieve a specific task with the task_id
n_tasks = task_suite.get_num_tasks() # number of tasks in the selected benchmark
print(f"[info] The benchmark {TASK_NAME} has {n_tasks} tasks.")


# retrieve a random task
#for i in range(n_tasks):
task_id = 7 # id of the task to retrieve
task = task_suite.get_task(task_id) # task is the object with all the information about the specific task
task_name = task.name # the name of the task as "KITCHEN_SCENE_1_put_the_black_bowl_at_the_front_on_the_coffee_table"
task_description = task.language # instruction in natural language ("Es. put the black bowl at the front on the coffee table")
print(f"\n[info] retrieving task {task_id} from suite {task_suite_name}\n")
print(f"[info] task description: {task_description}\n")

# retrieve the BDDL (Behavioral Design Definition Language) file for the task, it is a file containing all the informations about objects and initial and goal state
# (Ex. On Bowl1 Table1 means that the bowl 1 is on the table at initial state for example)
# get_libero_path("bddl_files"): find the directory where BDDL files are stored in your pc (Ex. /home/user/Desktop/LIBERO/libero/libero/./bddl_files )
# task.problem_folder: the folder containing the BDDL file for the specific task (Ex. libero_spatial)
# task.bddl_file: the name of the BDDL file for the specific task (Ex. pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate.bddl)
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
print(f"[info] BDDL file for the task: {task_bddl_file}\n")


# Args for environment initialization
env_args = {
    "bddl_file_name": task_bddl_file, # path of the BDDL file
    "camera_heights": 128,
    "camera_widths": 128,
    "camera_names": ["agentview"],
    "has_renderer": False, # If True, open the MuJoCo screen to render the env
    "has_offscreen_renderer": True, # If True, save images rendered to create a video
    "use_camera_obs": True, # If True, the "obs" will include camera observations (e.g., RGB images) that can be used to create a video.
    "render_camera": RENDER_CAMERA, # the camera name used for rendering and saving video
    "controller": "OSC_POSE",
    "control_freq": 20
}

env = ControlEnv(**env_args) # create the env class
env.seed(0) # set a seed for reproducibility
env.reset() # reset the scene and bring to initial state

# init_states is a list of 50 possible different intial states
# each init_state is a vector of state formed by joint positions, objects positions and orientations, velocities ...
init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = random.randint(0, 49) # Among all 50 initial states you can spawn objects in different ways choosing init_state_id (it can be a int number in [0,49])
env.set_init_state(init_states[init_state_id]) # set the init_state chosen

# Invece di env.action_space, usa:
print(f"Controller type: {env.env.robots[0].controller.name}")
print("="*50 + "\n")

window_size = model.num_frames # the number of frames needed to make inference with the loaded model ActionJEPA

interpolation_dict= {
    "nearest": cv2.INTER_NEAREST,
    "linear": cv2.INTER_LINEAR,
    "area": cv2.INTER_AREA,
    "cubic": cv2.INTER_CUBIC
}

# the get method return the value corresponding to the key INTERPOLATION TYPE, otherwise it return INTER CUBIC method
interpolation = interpolation_dict.get(INTERPOLATION_TYPE, cv2.INTER_CUBIC)

frame_buffer = deque(maxlen=window_size) # the frame buffer for making inference with ActionJEPA
joint_buffer = deque(maxlen=model.T)

# Define the text instruction for the VLA
text_input = task_description

print(f"Task: {task_description}")

init_action = [0.] * 7 # a start action needed for the first step, to have the first observation from the environment
obs, _, _, _ = env.step(init_action) # a first step with a zero action to observe the first frame

target_joints = joint_states[0, :7] 

# 2. Forza il simulatore a mettere il robot esattamente in quella posizione
# In Robosuite/LIBERO si fa così:
env.env.robots[0].set_robot_joint_positions(target_joints)

# 3. Fai un piccolo step a vuoto per aggiornare la fisica

for _ in range(window_size):
    
    raw_frame = obs["agentview_image"]
    saving_frame = np.flip(raw_frame, axis=0) 
    saving_frame = cv2.resize(saving_frame, (256, 256), interpolation=interpolation)
    frame_buffer.append(saving_frame)
    
    current_joints = obs["robot0_joint_pos"] 
    joint_buffer.append(current_joints)
    
    obs, _, _, _ = env.step(init_action)

video_frames = [] # list used to store frames for video saving

# Loop over the environment to apply actions and collect observations
# Obs is an ordered dict that have information about the ROBOT:
# - "agentview_image": the RGB image from the agent's camera => camera_heights x camera_widths x 3 (ex 1024 x 1024 x 3)
# - "robot0_joint_pos": array of vlues of the robot's joint positions
# - "robot0_joint_pos_cos (or sin)": array of cos (or sin) values of the robot's joint positions
# - "robot0_joint_vel": array of values of the robot's joint velocities
# - "robot0_eef_pos": array of values of the robot's end effector position (x,y,z)
# - "robot0_eef_quat": array of values of the robot's end effector orientation in quaternion (x,y,z,w)
# - "robot0_gripper_qpos": the gripper's position (two values for the two fingers distances from the center, 0 means fully closed)
# - "robot0_gripper_qvel": the gripper's velocity (two values for the two fingers velocities)
# And information about the object. Each object called "objectName" has two arrays:
# - "objectName_pos": array of values of the object's position (x,y,z)
# - "objectName_quat": array of values of the object's orientation in quaternion (x,y,z,w)
# - "objectName_to_robot0_eef_pos": array of values of the object's position relative to the robot's end effector (x,y,z)
# - "objectName_to_robot0_eef_quat": array of values of the object's orientation relative to the robot's end effector (x,y,z,w)

with torch.no_grad():
    z_tokens = model.language_backbone.tokenization(text_input)
    text_input = model.language_backbone(z_tokens).float()
    for step in tqdm(range(window_size, 130), desc="🤖 Inference execution"):

        current_frame = np.flip(obs["agentview_image"], axis=0) # get the RGB image from the agent's camera
        current_frame = cv2.resize(current_frame, (256, 256), interpolation=interpolation)
        current_joints = obs["robot0_joint_pos"]
        print(current_joints)
        
        saving_frame = cv2.resize(np.flip(obs["agentview_image"], axis=0), (256, 256), interpolation=interpolation)
        
        video_frames.append(saving_frame) # needed to save the video
        
        frame_buffer.append(current_frame) # add the current frame to the frame buffer
        joint_buffer.append(current_joints)

        vision_input = list(frame_buffer)
        joint_input = torch.from_numpy(np.array(joint_buffer)).float().unsqueeze(0).to(device)

        z_frames = model.vision_backbone.preprocess_frames(vision_input)
        vision_input = model.vision_backbone(z_frames).float()

        model.use_backbone = False
        
        actor_action_seq, refiner_action_seq = model(text_input, vision_input, joint_input)

        vla_action = actor_action_seq[0, 0, :].cpu().numpy()
    
        obs, reward, done, _ = env.step(vla_action)

        if done:
            print(f"Task completed at step: {step}")
            break

    # At the end the env is closed
    env.close()

    # Save video if required to a task.mp4 file
    videoname = f"{task_suite_name}_{task_id}_{task_description.replace(' ', '_')}.mp4"
    if len(video_frames) > 0:
        imageio.mimsave(os.path.join(results_dir_path, videoname), video_frames, fps=60)
        print(f"Video saved: {os.path.join(results_dir_path, videoname)}")


[info] RENDER_CAMERA: agentview

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] The benchmark libero_goal has 10 tasks.

[info] retrieving task 7 from suite libero_goal

[info] task description: turn on the stove

[info] BDDL file for the task: /home/cyberm/Desktop/LIBERO/libero/libero/./bddl_files/libero_goal/turn_on_the_stove.bddl

[Warning]: datasets path /home/cyberm/Desktop/LIBERO/libero/libero/../datasets does not exist!
Controller type: OSC_POSE

Task: turn on the stove


🤖 Inference execution:   0%|          | 0/126 [00:00<?, ?it/s]

[-2.08715238e-02 -1.42191424e-01 -1.13765428e-02 -2.41787233e+00
  6.46794465e-04  2.20595952e+00  7.80469803e-01]
[-2.03066815e-02 -1.32508168e-01 -1.34628649e-02 -2.42261976e+00
  6.67649951e-04  2.22449486e+00  7.71210639e-01]
[-2.01844142e-02 -1.24914612e-01 -1.69967822e-02 -2.42792010e+00
  9.57615209e-04  2.23913793e+00  7.64299401e-01]
[-1.99794196e-02 -1.24749895e-01 -1.99238383e-02 -2.43050952e+00
  7.86665094e-04  2.24151317e+00  7.60357939e-01]
[-2.00380974e-02 -1.19856083e-01 -2.36995114e-02 -2.43176053e+00
  5.89684182e-04  2.24636333e+00  7.57395329e-01]
[-2.04238181e-02 -1.12226279e-01 -2.85017561e-02 -2.43284085e+00
 -6.56069624e-04  2.25325106e+00  7.55202568e-01]
[-0.0206808  -0.10741516 -0.03278031 -2.43298838 -0.00285243  2.25671028
  0.75141209]
[-0.02104137 -0.10369403 -0.03725741 -2.43220589 -0.00558774  2.2574258
  0.74736846]
[-0.02175454 -0.09851928 -0.04285814 -2.43104512 -0.00770897  2.25814789
  0.74305745]
[-0.02269598 -0.09242093 -0.04882383 -2.42848401 -